In [8]:
%matplotlib qt5
import hyperspy.api as hs
import pyxem as pxm
import numpy as np
import orix
import matplotlib.pyplot as plt

from pathlib import Path
from diffpy.structure import Atom, Lattice, Structure
from diffsims.generators.simulation_generator import SimulationGenerator
from orix.vector import Miller, Vector3d
from orix.plot import IPFColorKeyTSL

In [2]:
PATH_SPED5 = Path("D:/Datasets/250225_Ag_oxww/20250225_175525/SPED5_calibrated.zspy")
PATH_SPED6 = Path("D:/Datasets/250225_Ag_oxww/20250225_180346/SPED6_calibrated.zspy")
PATH_SPED7 = Path("D:/Datasets/250225_Ag_oxww/20250225_181109/SPED7_calibrated.zspy")
PATH_SPED8 = Path("D:/Datasets/250225_Ag_oxww/20250225_182247/SPED8_calibrated.zspy")

def masking(path, disk_radius=3, threshold=0.6, mark=False):
    dataset = hs.load(path, lazy=True)
    dataset.change_dtype("float32")

    ny, nx = dataset.axes_manager.signal_shape
    center = (ny // 2, nx // 2)
    dataset.calibration(center=center)

    template = dataset.template_match_disk(disk_r=disk_radius, subtract_min=False)
    peaks = template.get_diffraction_vectors(min_distance=3, threshold_abs=threshold)
    if mark:
        markers = peaks.to_markers(sizes=3, colors="red")
        mask = peaks.to_mask(disk_r=disk_radius)
        dataset = dataset * mask
        return dataset, markers
    else:
        mask = peaks.to_mask(disk_r=disk_radius)
        dataset = dataset * mask
        return dataset

dataset = masking(PATH_SPED7)
ix, iy = dataset.axes_manager.indices
dataset.plot()

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5487)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5492)


  0%|          | 0/25 [00:00<?, ?it/s]

In [3]:
a = 4.08
atoms = [Atom('Ag', [0, 0, 0]), Atom('Ag', [0.5, 0.5, 0]), Atom('Ag', [0.5, 0, 0.5]), Atom('Ag', [0, 0.5, 0.5])]
lattice = Lattice(a, a, a, 90, 90, 90)
phase = orix.crystal_map.Phase(name='Ag', space_group=225, structure=Structure(atoms, lattice))

angular_resolution = 0.5
grid = orix.sampling.get_sample_reduced_fundamental(angular_resolution, point_group=phase.point_group)
orientations = orix.quaternion.Orientation(grid, symmetry=phase.point_group)

simgen = SimulationGenerator(precession_angle=1., minimum_intensity=1e-5)
simulations = simgen.calculate_diffraction2d(phase=phase, rotation=grid, reciprocal_radius=1.5, with_direct_beam=False, max_excitation_error=0.01)

In [ ]:
def orientation_function(data):
    azi_integral = data.get_azimuthal_integral2d(npt=256)
    results = azi_integral.get_orientation(simulations, n_best=grid.size, frac_keep=0.5)
    results.compute() 
    return results

results = orientation_function(dataset)

WARNING | Hyperspy | The function you applied does not take into account the difference of scales in-between axes. (hyperspy.signal:5487)
WARNING | Hyperspy | The function you applied does not take into account the difference of units in-between axes. (hyperspy.signal:5492)


  0%|          | 0/19 [00:00<?, ?it/s]

In [ ]:
def plot_ipf(result_data, ix, iy):
    correlations_i = result_data.inav[ix, iy].data[:, 1]
    template_indices_i = (result_data.inav[ix, iy].data[:, 0]).astype('int16')
    orientations_i = orientations[template_indices_i]

    fig = plt.figure()
    ax = fig.add_subplot(111, projection="ipf", symmetry=phase.point_group)
    ax.scatter(orientations_i, c=correlations_i, cmap='inferno')
    ax.set_title(f"IPF at pixel ({ix}, {iy})")
    plt.show()

plot_ipf(results, 0, 0)

In [9]:

xmap = results.to_crystal_map()
oris = xmap.orientations
corrs = results.data[:,:,0,1].flatten()

key_x = IPFColorKeyTSL(phase.point_group, Vector3d.xvector())
key_y = IPFColorKeyTSL(phase.point_group, Vector3d.yvector())
key_z = IPFColorKeyTSL(phase.point_group, Vector3d.zvector())

oris_z = key_z.orientation2color(oris)
xmap.plot(oris_z, overlay=corrs, remove_padding=True)
oris_x = key_x.orientation2color(oris)
xmap.plot(oris_x, overlay=corrs, remove_padding=True)
oris_y = key_y.orientation2color(oris)
xmap.plot(oris_y, overlay=corrs, remove_padding=True)


  0%|          | 0/65 [00:00<?, ?it/s]

In [ ]:
def iterate_x(Orientations_list, x_max):
    pixel_misorientation_matrix = []
    pixel_misorientation_list = []
    for i in range(Orientations_list.shape[0]):
        # start a new row whenever we hit a multiple of x_max
        if i % x_max == 0 and i != 0:
            pixel_misorientation_matrix.append(pixel_misorientation_list)
            pixel_misorientation_list = []

        # if we are not at the last column in this row
        if i % x_max != x_max - 1 and i != Orientations_list.shape[0]:
            # if left, me, and right are identical orientations → put 0
            if (Orientations_list[i-1] == Orientations_list[i]
                and Orientations_list[i] == Orientations_list[i+1]):
                pixel_misorientation_list.append(0)
            else:
                # otherwise compute misorientation to pixel on the right
                pixel_misorientation_list.append(
                    Orientations_list[i].angle_with_outer(
                        Orientations_list[i+1], degrees=True
                    )[0][0]
                )
        else:
            # last pixel in the row, no right neighbour → 0
            pixel_misorientation_list.append(0)

    pixel_misorientation_matrix.append(pixel_misorientation_list)
    return pixel_misorientation_matrix

def iterate_y(Orientations_list, x_max, y_max, pixel_misorientation_matrix):
    for i in range(Orientations_list.shape[0]):
        if i + x_max < Orientations_list.shape[0]:
            # if at least one of (above, me, below) is different
            if (Orientations_list[i - x_max] != Orientations_list[i]
                or Orientations_list[i] != Orientations_list[i + x_max]):

                current_val = Orientations_list[i].angle_with_outer(
                    Orientations_list[i + x_max], degrees=True
                )[0][0]

                # pick the LARGEST of: "x-direction misori we already had"
                # vs "y-direction misori we just computed"
                if current_val > pixel_misorientation_matrix[i // x_max][i % x_max]:
                    pixel_misorientation_matrix[i // x_max][i % x_max] = current_val
    return pixel_misorientation_matrix

def grain_boundaries_misorientations(Orientations_list, x_max, y_max):
    pixel_misorientation_matrix = iterate_x(Orientations_list, x_max)
    pixel_misorientation_matrix = iterate_y(
        Orientations_list, x_max, y_max, pixel_misorientation_matrix
    )
    return np.array(pixel_misorientation_matrix)



In [ ]:
def iterate_x_updated(Orientations_list, x_max, ix, iy):
    pixel_misorientation_matrix = []
    pixel_misorientation_list = []
    for i in range(Orientations_list.shape[0]):
        # start a new row whenever we hit a multiple of x_max
        if i % x_max == 0 and i != 0:
            pixel_misorientation_matrix.append(pixel_misorientation_list)
            pixel_misorientation_list = []

        # if we are not at the last column in this row
        if i % x_max != x_max - 1 and i != Orientations_list.shape[0]:
            # if left, me, and right are identical orientations → put 0
            if (Orientations_list[i] == Orientations_list[ix + iy * x_max]):
                pixel_misorientation_list.append(0)
            else:
                # otherwise compute misorientation to pixel on the right
                pixel_misorientation_list.append(
                    Orientations_list[i].angle_with_outer(
                        Orientations_list[ix + iy * x_max], degrees=True
                    )[0][0]
                )
        else:
            # last pixel in the row, no right neighbour → 0
            pixel_misorientation_list.append(0)

    pixel_misorientation_matrix.append(pixel_misorientation_list)
    return pixel_misorientation_matrix

def iterate_y_updated(Orientations_list, x_max, y_max, pixel_misorientation_matrix, ix, iy):
    for i in range(Orientations_list.shape[0]):
        if i + x_max < Orientations_list.shape[0]:
            # if at least one of (above, me, below) is different
            if (Orientations_list[i - x_max] != Orientations_list[i]
                or Orientations_list[i] != Orientations_list[i + x_max]):

                current_val = Orientations_list[i].angle_with_outer(
                    Orientations_list[ix + iy * x_max], degrees=True
                )[0][0]

                # pick the LARGEST of: "x-direction misori we already had"
                # vs "y-direction misori we just computed"
                if current_val > pixel_misorientation_matrix[i // x_max][i % x_max]:
                    pixel_misorientation_matrix[i // x_max][i % x_max] = current_val
    return pixel_misorientation_matrix

def grain_boundaries_misorientations_updated(Orientations_list, x_max, y_max, ix, iy):
    pixel_misorientation_matrix = iterate_x_updated(Orientations_list, x_max, ix, iy)
    pixel_misorientation_matrix = iterate_y_updated(
        Orientations_list, x_max, y_max, pixel_misorientation_matrix, ix, iy
    )
    return np.array(pixel_misorientation_matrix)

In [ ]:
misoriented_data = grain_boundaries_misorientations(oris, 200, 100)

plt.figure(figsize=(6,5))
img = plt.imshow(misoriented_data)
plt.colorbar(img)
plt.title("Misorientations")
plt.show()